# Cost of the LMM runs (Appendix: Cost Analysis)

What it would cost to run each model **once** over the evaluation: every
question asked one time, per condition (original and `archive-light`), plus
Gemini's run on the published figures.

**Gemini and Claude** -- exact, from the token counts each API response
recorded (`usage` in every answer of the run pickles).  Each stored answer
carries the usage of the single call that produced it, so earlier test runs
and discarded attempts never enter; a question that was re-asked (the
provenance fix) is counted once.

**GPT** -- its responses did not record token usage, and it ran at default
reasoning, whose reasoning tokens are billed but not returned, so the cost
cannot be rebuilt from the answers.  It is scaled instead from the OpenAI usage
dashboard (`platform.openai.com/usage`): the billed cost of the last GPT runs,
which sent only the 105 figures added to grow the sample from 150 to 255, in
both conditions.  The number of calls those runs made is counted from the run
notebooks' own output, and the cost per call is scaled to a full run.

Prices are list prices, standard tier, per million tokens -- check them and
update `prices_checked` when the paper is finalised.  The LaTeX table for the
appendix is printed at the end and written to `paper_figures/table_cost.tex`.

In [1]:
# ======================== CONFIG ========================
data_root = '~/astro_sky_image_vqa/'
runs = {                                      # condition -> run directory
    'original':          data_root + 'LMM_outputs_n255/',
    'archive-light':     data_root + 'LMM_outputs_n255_archive_light/',
    'published figures': data_root + 'LMM_outputs_realskyimages/',
}
model_dirs = {'gemini': 'Gemini', 'claude_haiku': 'Claude', 'chatgpt_api': 'GPT'}

# USD per million tokens (input, output), standard tier
prices = {
    'GPT':    (0.20, 1.25),    # gpt-5.4-nano          -- developers.openai.com/api/docs/pricing
    'Gemini': (0.30, 2.50),    # gemini-3.5-flash-lite  -- ai.google.dev/gemini-api/docs/pricing
    'Claude': (1.00, 5.00),    # claude-haiku-4-5       -- Anthropic model pricing
}
prices_checked = '12 October 2026'

# ---- GPT: from the OpenAI usage dashboard ----
# Billed cost on 2026-10-09 (UTC; the dashboard's day boundary) -- the runs of
# models/test_models/chatgpt_withapi_test.ipynb and ..._archive_light.ipynb,
# which executed 00:27-03:32 UTC that day and sent only the figures not
# already done (105 of 255, in each condition).  Assumes nothing else ran
# against gpt-5.4-nano that day.
gpt_dashboard_cost = 3.34
gpt_dashboard_runs = ['../test_models/chatgpt_withapi_test.ipynb',
                      '../test_models/chatgpt_withapi_test_archive_light.ipynb']
# Exact token totals for the same window, if ever available, take precedence
# (None = not available).
gpt_dashboard_tokens = {'input_tokens': None, 'output_tokens': None, 'requests': None}

tab_out = '~/Dropbox/WASP2026/paper_figures/table_cost.tex'

In [2]:
import os, glob, pickle
import pandas as pd

ex = os.path.expanduser


def usage_tokens(u):
    """(input, output) tokens from a stored usage record (dict or SDK object)."""
    get = (lambda k: u.get(k)) if isinstance(u, dict) else (lambda k: getattr(u, k, None))
    return (get('input_tokens') or 0), (get('output_tokens') or 0)


rows = []
for cond, rdir in runs.items():
    for mdir, mname in model_dirs.items():
        files = sorted(glob.glob(os.path.join(ex(rdir), mdir, '*_qa.pickle')))
        if not files:
            continue
        calls = with_usage = tin = tout = 0
        for fp in files:
            payload = pickle.load(open(fp, 'rb'))
            for e in (payload[0] if isinstance(payload, (list, tuple)) else payload):
                calls += 1
                if e.get('usage') is not None:
                    i, o = usage_tokens(e['usage'])
                    with_usage += 1; tin += i; tout += o
        rows.append({'model': mname, 'condition': cond, 'figures': len(files), 'calls': calls,
                     'calls with usage': with_usage, 'input tokens': tin, 'output tokens': tout})

usage = pd.DataFrame(rows)
usage

,model,condition,figures,calls,calls with usage,input tokens,output tokens
0,Gemini,original,255,6035,6035,7544741,862559
1,Claude,original,255,6035,6035,10342841,958896
2,GPT,original,255,6035,0,0,0
3,Gemini,archive-light,255,6035,6035,7543517,767386
4,Claude,archive-light,255,6035,6035,10345730,952765
5,GPT,archive-light,255,6035,0,0,0
6,Gemini,published figures,104,520,520,655757,195831


## Cost per run

Exact for every (model, condition) whose answers all recorded usage.  A run
with no recorded usage (GPT) is filled from the dashboard cell below.

In [3]:
def cost(model, tin, tout):
    pin, pout = prices[model]
    return tin * pin / 1e6 + tout * pout / 1e6


exact = usage[usage['calls with usage'] == usage['calls']].copy()
exact['cost $'] = [cost(m, i, o) for m, i, o in zip(exact.model, exact['input tokens'], exact['output tokens'])]
exact['$ per call'] = exact['cost $'] / exact['calls']

missing = usage[usage['calls with usage'] < usage['calls']]
if len(missing):
    print('no (or partial) recorded usage -- needs the dashboard:',
          ', '.join('%s / %s (%d of %d calls)' % (r.model, r.condition, r['calls with usage'], r.calls)
                    for _, r in missing.iterrows()))
exact.round({'cost $': 2, '$ per call': 5})

no (or partial) recorded usage -- needs the dashboard: GPT / original (0 of 6035 calls), GPT / archive-light (0 of 6035 calls)


,model,condition,figures,calls,calls with usage,input tokens,output tokens,cost $,$ per call
0,Gemini,original,255,6035,6035,7544741,862559,4.42,0.00073
1,Claude,original,255,6035,6035,10342841,958896,15.14,0.00251
3,Gemini,archive-light,255,6035,6035,7543517,767386,4.18,0.00069
4,Claude,archive-light,255,6035,6035,10345730,952765,15.11,0.00250
6,Gemini,published figures,104,520,520,655757,195831,0.69,0.00132


## GPT, from the dashboard

Average tokens per request over the dashboard window, scaled to the calls one
run makes (6035 per condition, both conditions).  The dashboard window also
holds test runs and re-asks; averaging per request is what removes them.

In [4]:
import json, re

gpt_calls = int(usage[(usage.model == 'GPT') & usage.condition.isin(['original', 'archive-light'])]['calls'].sum())

# Calls the dashboard window covered: figures each run notebook actually sent
# (all figures minus those it reports as 'have file already'), counted in
# questions from the run pickles.
covered = 0
for nb_path, cond in zip(gpt_dashboard_runs, ('original', 'archive-light')):
    nb = json.load(open(nb_path))
    out = '\n'.join(''.join(o.get('text', '')) for c in nb['cells'] for o in c.get('outputs', []) if 'text' in o)
    skipped = set(re.findall(r'have file already: \S*?(vqa_\d{6})', out))
    rdir = os.path.join(ex(runs[cond]), 'chatgpt_api')
    sent = [p for p in glob.glob(os.path.join(rdir, '*_qa.pickle'))
            if os.path.basename(p).removesuffix('_qa.pickle') not in skipped]
    n = sum(len(pickle.load(open(p, 'rb'))[0]) for p in sent)
    covered += n
    print('%-14s %3d figures sent (%d already done) -> %d calls' % (cond, len(sent), len(skipped), n))

if all(v is not None for v in gpt_dashboard_tokens.values()):
    per_in = gpt_dashboard_tokens['input_tokens'] / gpt_dashboard_tokens['requests']
    per_out = gpt_dashboard_tokens['output_tokens'] / gpt_dashboard_tokens['requests']
    gpt_row = {'input tokens': per_in * gpt_calls, 'output tokens': per_out * gpt_calls}
    gpt_row['cost $'] = cost('GPT', gpt_row['input tokens'], gpt_row['output tokens'])
    basis = 'dashboard tokens per request'
else:
    per_call = gpt_dashboard_cost / covered
    gpt_row = {'input tokens': None, 'output tokens': None, 'cost $': per_call * gpt_calls}
    basis = '$%.2f billed for %d calls = $%.6f per call' % (gpt_dashboard_cost, covered, per_call)
gpt_row.update({'model': 'GPT', 'condition': 'original + archive-light', 'calls': gpt_calls,
                '$ per call': gpt_row['cost $'] / gpt_calls})
print('GPT, one run over both conditions (%d calls): $%.2f   [%s]' % (gpt_calls, gpt_row['cost $'], basis))

original       105 figures sent (150 already done) -> 2485 calls
archive-light  105 figures sent (150 already done) -> 2485 calls
GPT, one run over both conditions (12070 calls): $8.11   [$3.34 billed for 4970 calls = $0.000672 per call]


## The appendix table

Rows: GPT as one total over both conditions (scaled); Gemini and Claude per
condition (exact); then Gemini's published-figure run.  Paste the printed
table over the one in the paper after any change.

In [5]:
def m_tok(x):
    return '%.2f' % (x / 1e6)


lines = []
tok = lambda x: m_tok(x) if x is not None else '--'
lines.append(r'GPT & both (est.) & %s & %s & %.2f \\' % (tok(gpt_row['input tokens']),
                                                     tok(gpt_row['output tokens']), gpt_row['cost $']))
for model in ('Gemini', 'Claude'):
    for cond in ('original', 'archive-light'):
        r = exact[(exact.model == model) & (exact.condition == cond)].iloc[0]
        lines.append(r'%s & %s & %s & %s & %.2f \\' % (model, cond, m_tok(r['input tokens']),
                                                      m_tok(r['output tokens']), r['cost $']))
lines.append(r'\midrule')
r = exact[(exact.model == 'Gemini') & (exact.condition == 'published figures')].iloc[0]
lines.append(r'Gemini & published & %s & %s & %.2f \\' % (m_tok(r['input tokens']), m_tok(r['output tokens']), r['cost $']))

price_note = '; '.join('%s \\$%.2f / \\$%.2f' % (m, *prices[m]) for m in ('GPT', 'Gemini', 'Claude'))
table = r"""\begin{table}[h]
\centering\small\color{red}
\begin{tabular}{@{}llrrr@{}}
\toprule
Model & Condition & In (M) & Out (M) & Cost (\$) \\
\midrule
%s
\bottomrule
\end{tabular}
\caption{Cost of one run of each model: tokens in and out (millions) and cost in USD at list prices per million input / output tokens (%s), checked %s. Each condition is %d calls per model; the published-figure run is %d calls (\textsf{Gemini} only). GPT, whose responses do not record token usage, is scaled from its provider's billed cost for the last %d calls (both conditions).}
\label{tab:cost}
\end{table}""" % ('\n'.join(lines), price_note, prices_checked,
                  int(exact[exact.condition == 'original']['calls'].iloc[0]),
                  int(exact[exact.condition == 'published figures']['calls'].iloc[0]), covered)

open(ex(tab_out), 'w').write(table + '\n')
print(table)

\begin{table}[h]
\centering\small\color{red}
\begin{tabular}{@{}llrrr@{}}
\toprule
Model & Condition & In (M) & Out (M) & Cost (\$) \\
\midrule
GPT & both (est.) & -- & -- & 8.11 \\
Gemini & original & 7.54 & 0.86 & 4.42 \\
Gemini & archive-light & 7.54 & 0.77 & 4.18 \\
Claude & original & 10.34 & 0.96 & 15.14 \\
Claude & archive-light & 10.35 & 0.95 & 15.11 \\
\midrule
Gemini & published & 0.66 & 0.20 & 0.69 \\
\bottomrule
\end{tabular}
\caption{Cost of one run of each model: tokens in and out (millions) and cost in USD at list prices per million input / output tokens (GPT \$0.20 / \$1.25; Gemini \$0.30 / \$2.50; Claude \$1.00 / \$5.00), checked 12 October 2026. Each condition is 6035 calls per model; the published-figure run is 520 calls (\textsf{Gemini} only). GPT, whose responses do not record token usage, is scaled from its provider's billed cost for the last 4970 calls (both conditions).}
\label{tab:cost}
\end{table}
